# 00 | HMI Data Contract with real BoneNet signals

**Human Movement Intelligence | Python-first research pipeline**  
**Presented by: Reza Saadatyar (2026)**

**Objective:** Load the public authors' BoneNet derivative with the existing pilot reader, wrap it in the reusable `hmi.data.DatasetBundle`, inspect its structural summary and plot actual signals. No model is trained.

**Limits:** The source-to-label row order and source strain units are not independently verified.

## Step 1 | Imports and project environment

Find the repository root from the notebook folder and import the Python package.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from hmi.data import DatasetBundle

repo_root = Path.cwd().resolve()
while not (repo_root / 'pyproject.toml').is_file():
    if repo_root.parent == repo_root:
        raise RuntimeError('Open inside the human-movement-intelligence repository')
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from scripts.bonenet_python_pilot import fetch_file, inspect_source
print('Repository:', repo_root)

## Step 2 | Download public numeric derivative

The raw files are cached under the ignored `data/raw/bonenet_reference/` folder. This is a study-specific source adapter, not the universal HMI contract.

In [ ]:
data_root = repo_root / 'data' / 'raw' / 'bonenet_reference'
for filename in ('Data.mat', 'labels.csv'):
    item = fetch_file(filename, data_root)
    print(f"{filename}: {item['retrieval']} ({item['bytes']:,} bytes)")

## Step 3 | Decode and inspect real source arrays

The existing BoneNet pilot turns the authors' MATLAB numeric `inputs` and `outputs` into arrays. It is not a general MAT reader.

In [ ]:
source_audit, label_rows, x, y = inspect_source(data_root)
if x is None or y is None:
    raise ValueError(f"Numeric data unavailable: {source_audit['blockers']}")
print('X.shape:', x.shape, 'Y.shape:', y.shape)
print('Subject IDs:', source_audit['labels']['subjects'])
print('Independently verified labels:', source_audit['labels']['source_row_order_independently_verified'])

## Step 4 | Construct the reusable DatasetBundle

`sample_ids` are local row identifiers; they are **not confirmed original trial IDs**.

In [ ]:
bundle = DatasetBundle(
    dataset_id='bonenet_derivative',
    inputs={'imu': x},
    targets={'strain': y},
    subject_ids=[r['SubjID'].strip().lower() for r in label_rows],
    sample_ids=[f'bonenet_row_{i:04d}' for i in range(len(label_rows))],
    input_axes={'imu': ('sample', 'channel', 'time')},
    target_axes={'strain': ('sample', 'time', 'target')},
    units={'imu': 'unverified source units', 'strain': 'unverified source units'},
    provenance={'source': 'authors BoneNet derivative',
                'sample_label_alignment_verified': False,
                'sample_ids': 'local row indices only', 'targets': ['e1', 'e3']}
)
profile = bundle.summary()
print(json.dumps(profile, indent=2))

## Step 5 | View structured output

This compact table will later be replaced with a richer, modality-aware profiler.

In [ ]:
rows = [{'section': section, 'name': name, **values}
        for section in ('inputs', 'targets')
        for name, values in profile[section].items()]
display(pd.DataFrame(rows))
print(f'{bundle.n_samples} records across {bundle.n_subjects} subject IDs')

## Step 6 | Plot real IMU and modelled strain sequences

These plots use real source values, without making claims about absolute physical units or validated subject pairing.

In [ ]:
record_index, channel_index = 0, 0
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
axes[0].plot(bundle.inputs['imu'][record_index, channel_index])
axes[0].set(title='IMU: example record and channel', xlabel='Normalised cycle point', ylabel='Source value (units unverified)')
axes[1].plot(bundle.targets['strain'][record_index, :, 0], label='e1')
axes[1].plot(bundle.targets['strain'][record_index, :, 1], label='e3')
axes[1].set(title='Model-derived FE target sequences', xlabel='Normalised cycle point', ylabel='Source value (units unverified)')
axes[1].legend()
fig.tight_layout()
plt.show()

## Step 7 | Contract quality gate

Passing the structural checks does not prove row-order linkage, trustworthy physical units, absence of leakage or clinical validity.

In [ ]:
assert bundle.n_samples == len(label_rows)
assert bundle.inputs['imu'].shape[0] == bundle.targets['strain'].shape[0]
assert not bundle.provenance['sample_label_alignment_verified']
print('Structural contract: PASS')
print('Independent sample-label alignment: NOT VERIFIED')
print('Training performed: NO')